In [2]:
import pandas as pd
import numpy as np
import joblib

data = joblib.load("../data/dev_questions_with_search_results_and_prediction.joblib")
data.head()

,Question_ID,Domain,N_Pages,Question,A,B,C,D,E,F,...,reranked_retrieved_docs,recall_at_1,recall_at_3,recall_at_5,recall_at_10,correct_doc,page_score,retrieved_docs_simplified,pred_answer,answer_is_correct
0,0,domain_2,5,Як рекомендовано приймати ретаболіл дорослим?,внутрішньо,підшкірно,орально,внутрішньовенно,внутрішньом’язово,інгаляційно,...,"[{'page_number': 1, 'doc_id': '4e779acee13fa6e...",1,1,1,1,1,1.000000,"[{'page_number': 1, 'doc_id': '4e779acee13fa6e...",E,True
1,1,domain_1,89,Які типи змагань проводяться з боротьби самбо?,тільки індивідуальні,"особисті, командні, особисто-командні",тільки особисті,змагання не проводяться,тільки командні,тільки командні та індивідуальні,...,"[{'page_number': 3, 'doc_id': '72e790ea927cb79...",1,1,1,1,1,1.000000,"[{'page_number': 3, 'doc_id': '72e790ea927cb79...",B,True
2,2,domain_2,15,На підставі якого фактору слід визначати доціл...,Тяжкість інфекційного процесу,Особистий досвід лікаря,Виключно за результатами аналізів,Залежно від пацієнта,Наявність алергічних реакцій,Літній вік,...,"[{'page_number': 7, 'doc_id': '1caa31f28b0feab...",1,1,1,1,1,1.000000,"[{'page_number': 7, 'doc_id': '1caa31f28b0feab...",A,True
3,3,domain_1,101,Яка тривалість тайм-ауту в баскетболі?,45 секунд,2 хвилини,3 хвилини,1 хвилина,"1,5 хвилини",30 секунд,...,"[{'page_number': 97, 'doc_id': 'b59848334218c3...",1,1,1,1,1,0.257426,"[{'page_number': 97, 'doc_id': 'b59848334218c3...",D,True
4,4,domain_2,8,Яку дію здійснює фенілраміну малеат в складі ф...,стимулює активність мікросомальних ферментів п...,компенсує потреби організму у вітаміні С,чинить протизапальну дію,знижує температуру,зменшує запальну реакцію слизових оболонок,знеболює,...,"[{'page_number': 1, 'doc_id': 'f4539bc3a790162...",1,1,1,1,1,1.000000,"[{'page_number': 1, 'doc_id': 'f4539bc3a790162...",E,True


# Search evaluation: 

In [3]:
def recall_at_k(results, ground_truth, k):
    """Compute Recall@K for a single query."""
    retrieved_ids = [doc["doc_id"] for doc in results[:k]]
    return int(ground_truth in retrieved_ids)

def recall_at_k_page(results, ground_truth, k):
    """Compute Recall@K for a single query."""
    results_tuples = [(r['doc_id'], r["page_number"]) for r in results[:k]]
    true_tuple = (ground_truth["Doc_ID"], ground_truth["Page_Num"])
    return int(true_tuple in results_tuples)

def page_score_np(page_pred, page_true, n_pages, d):
    """Compute the page score using numpy."""
    score = (1 - np.abs(page_pred - page_true) / n_pages) * d
    return score

def calculate_metrics_df(df, column="retrieved_docs"):
    # Calculating metrics:
    print("Document search scores: ")
    k_values = [1, 2, 3, 4, 5, 10]
    for k in k_values:
        df[f"recall_at_{k}"] = df.apply(lambda row: recall_at_k(row[column], row["Doc_ID"], k), axis=1)
        print(f"Recall@{k}: {df[f'recall_at_{k}'].mean():.4f}")

    print("\nPage search scores: ")
    k_values = [1, 2, 3, 4, 5, 10]
    for k in k_values:
        df[f"recall_at_{k}"] = df.apply(lambda row: recall_at_k_page(row[column], row, k), axis=1)
        print(f"Recall@{k}: {df[f'recall_at_{k}'].mean():.4f}")
    
    # Page score calculation:
    df["correct_doc"] = df.apply(
        lambda row: 1 if row[column][0]["doc_id"] == row["Doc_ID"] else 0, axis=1
    )
    df["page_score"] = df.apply(
        lambda row: page_score_np(
            row[column][0]["page_number"],
            row["Page_Num"],
            row["N_Pages"],
            row["correct_doc"],
        ),
        axis=1,
    )
    print(f"\nAverage Page Score: {df['page_score'].mean():.4f}")

In [6]:
calculate_metrics_df(data, column="reranked_retrieved_docs")

Document search scores: 
Recall@1: 0.9892
Recall@2: 0.9957
Recall@3: 0.9978
Recall@4: 0.9978
Recall@5: 1.0000
Recall@10: 1.0000

Page search scores: 
Recall@1: 0.7657
Recall@2: 0.8720
Recall@3: 0.9154
Recall@4: 0.9284
Recall@5: 0.9328
Recall@10: 0.9544

Average Page Score: 0.9405


# Lapa inference evaluation: 

In [7]:
data["answer_is_correct"].mean()

0.9240780911062907

# Error analysis: 